In [ ]:
TASK 3: STATE SPACE REPRESENTATION USING ALGORITHMS

In [ ]:
from collections import deque
import heapq
import time

In [ ]:
initial_state = {
    "time": 18,                  # 6 PM
    "peak_hour": True,
    "temperature": 35,
    "occupancy": 3,

    "ac": True,
    "refrigerator": True,

    "washing_machine": False,
    "dishwasher": False,
    "lights": True,

    "cost": 0
}

In [ ]:
def is_goal(state):
    return (
        state["washing_machine"] and
        state["dishwasher"]
    )

In [ ]:
def get_neighbors(state):

    neighbors = []

    # Washing Machine
    if not state["washing_machine"]:

        new_state = state.copy()
        new_state["washing_machine"] = True

        if state["peak_hour"]:
            new_state["cost"] += 15
        else:
            new_state["cost"] += 8

        neighbors.append(("Start Washing Machine", new_state))

    # Dishwasher
    if not state["dishwasher"]:

        new_state = state.copy()
        new_state["dishwasher"] = True

        if state["peak_hour"]:
            new_state["cost"] += 12
        else:
            new_state["cost"] += 6

        neighbors.append(("Start Dishwasher", new_state))

    # Lights OFF
    if state["lights"]:

        new_state = state.copy()
        new_state["lights"] = False

        neighbors.append(("Turn Lights OFF", new_state))

    # Advance Time

    if state["time"] < 22:

        new_state = state.copy()

        new_state["time"] += 1

        if new_state["time"] >= 20:
            new_state["peak_hour"] = False

        neighbors.append(("Advance Time", new_state))

    return neighbors

In [ ]:
neighbors = get_neighbors(initial_state)

for action, state in neighbors:
    print(action)
    print(state)
    print("-" * 40)

Start Washing Machine
{'time': 18, 'washing_machine': True, 'dishwasher': False, 'lights': True, 'cost': 15}
----------------------------------------
Start Dishwasher
{'time': 18, 'washing_machine': False, 'dishwasher': True, 'lights': True, 'cost': 12}
----------------------------------------
Turn Lights OFF
{'time': 18, 'washing_machine': False, 'dishwasher': False, 'lights': False, 'cost': 0}
----------------------------------------
Advance Time
{'time': 19, 'washing_machine': False, 'dishwasher': False, 'lights': True, 'cost': 0}
----------------------------------------


In [ ]:
#define BFS
def bfs(initial_state):
    queue = deque()

    # (current_state, path_taken)
    queue.append((initial_state, []))

    visited = set()
    nodes_explored = 0

    while queue:
        current_state, path = queue.popleft()
        nodes_explored += 1

        # Create a hashable representation of the state
        state_key = (
            current_state["time"],
            current_state["washing_machine"],
            current_state["dishwasher"],
            current_state["lights"]
        )

        if state_key in visited:
            continue

        visited.add(state_key)

        # Goal reached?
        if is_goal(current_state):
            return {
                "path": path,
                "final_state": current_state,
                "nodes_explored": nodes_explored
            }

        # Explore neighbors
        for action, neighbor in get_neighbors(current_state):
            queue.append((neighbor, path + [action]))

    return None

In [ ]:
#Run BFS
start = time.time()

bfs_result = bfs(initial_state)
bfs_time = time.time() - start
end = time.time()

print("===== BFS RESULT =====")

print("\nActions Taken:")
for step in result["path"]:
    print("->", step)

print("\nFinal State:")
print(result["final_state"])

print("\nTotal Cost:", result["final_state"]["cost"])

print("Nodes Explored:", result["nodes_explored"])

print("Execution Time:", round(end - start, 6), "seconds")

===== BFS RESULT =====

Actions Taken:
-> Advance Time
-> Advance Time
-> Start Dishwasher
-> Start Washing Machine

Final State:
{'time': 20, 'washing_machine': True, 'dishwasher': True, 'lights': True, 'cost': 14}

Total Cost: 14
Nodes Explored: 41
Execution Time: 0.000144 seconds


In [ ]:
#define DFS
def dfs(initial_state):
    stack = []

    # (current_state, path_taken)
    stack.append((initial_state, []))

    visited = set()
    nodes_explored = 0

    while stack:
        current_state, path = stack.pop()   # <-- Difference from BFS
        nodes_explored += 1

        state_key = (
            current_state["time"],
            current_state["washing_machine"],
            current_state["dishwasher"],
            current_state["lights"]
        )

        if state_key in visited:
            continue

        visited.add(state_key)

        if is_goal(current_state):
            return {
                "path": path,
                "final_state": current_state,
                "nodes_explored": nodes_explored
            }

        for action, neighbor in get_neighbors(current_state):
            stack.append((neighbor, path + [action]))

    return None

In [ ]:
#Run DFS
start = time.time()

dfs_result = dfs(initial_state)
dfs_time = time.time() - start
end = time.time()

print("===== DFS RESULT =====")

print("\nActions Taken:")
for step in result["path"]:
    print("->", step)

print("\nFinal State:")
print(result["final_state"])

print("\nTotal Cost:", result["final_state"]["cost"])

print("Nodes Explored:", result["nodes_explored"])

print("Execution Time:", round(end - start, 6), "seconds")

===== DFS RESULT =====

Actions Taken:
-> Advance Time
-> Advance Time
-> Start Dishwasher
-> Start Washing Machine

Final State:
{'time': 20, 'washing_machine': True, 'dishwasher': True, 'lights': True, 'cost': 14}

Total Cost: 14
Nodes Explored: 41
Execution Time: 0.00014 seconds


In [ ]:
def heuristic(state):

    h = 0

    if not state["washing_machine"]:
        h += 8

    if not state["dishwasher"]:
        h += 6

    if state["peak_hour"]:
        h += 5

    return h

In [ ]:
import heapq
from itertools import count

def astar(initial_state):

    counter = count()

    priority_queue = []

    heapq.heappush(
        priority_queue,
        (
            heuristic(initial_state),
            0,
            next(counter),
            initial_state,
            []
        )
    )

    visited = set()
    nodes_explored = 0

    while priority_queue:

        f, g, _, current_state, path = heapq.heappop(priority_queue)

        nodes_explored += 1

        state_key = (
            current_state["time"],
            current_state["washing_machine"],
            current_state["dishwasher"],
            current_state["lights"]
        )

        if state_key in visited:
            continue

        visited.add(state_key)

        if is_goal(current_state):
            return {
                "path": path,
                "final_state": current_state,
                "nodes_explored": nodes_explored
            }

        for action, neighbor in get_neighbors(current_state):

            g_new = neighbor["cost"]

            f_new = g_new + heuristic(neighbor)

            heapq.heappush(
                priority_queue,
                (
                    f_new,
                    g_new,
                    next(counter),
                    neighbor,
                    path + [action]
                )
            )

    return None

In [ ]:
#Run A*
start = time.time()

astar_result = astar(initial_state)
astar_time = time.time() - start

end = time.time()

print("===== A* RESULT =====")

print("\nActions Taken:")
for step in result["path"]:
    print("->", step)

print("\nFinal State:")
print(result["final_state"])

print("\nTotal Cost:", result["final_state"]["cost"])

print("Nodes Explored:", result["nodes_explored"])

print("Execution Time:", round(end - start, 6), "seconds")

===== A* RESULT =====

Actions Taken:
-> Advance Time
-> Advance Time
-> Start Dishwasher
-> Start Washing Machine

Final State:
{'time': 20, 'washing_machine': True, 'dishwasher': True, 'lights': True, 'cost': 14}

Total Cost: 14
Nodes Explored: 41
Execution Time: 0.000253 seconds


In [ ]:
#define Greedy BFS
import heapq
from itertools import count

def greedy_best_first_search(initial_state):

    counter = count()

    priority_queue = []

    heapq.heappush(
        priority_queue,
        (
            heuristic(initial_state),
            next(counter),
            initial_state,
            []
        )
    )

    visited = set()
    nodes_explored = 0

    while priority_queue:

        h, _, current_state, path = heapq.heappop(priority_queue)

        nodes_explored += 1

        state_key = (
            current_state["time"],
            current_state["washing_machine"],
            current_state["dishwasher"],
            current_state["lights"]
        )

        if state_key in visited:
            continue

        visited.add(state_key)

        if is_goal(current_state):
            return {
                "path": path,
                "final_state": current_state,
                "nodes_explored": nodes_explored
            }

        for action, neighbor in get_neighbors(current_state):

            heapq.heappush(
                priority_queue,
                (
                    heuristic(neighbor),
                    next(counter),
                    neighbor,
                    path + [action]
                )
            )

    return None

In [ ]:
#Run Greedy BFS
start = time.time()

greedy_result = greedy_best_first_search(initial_state)

greedy_time = time.time() - start

print("===== GREEDY BEST-FIRST SEARCH RESULT =====")

print("\nActions Taken:")
for step in greedy_result["path"]:
    print("->", step)

print("\nFinal State:")
print(greedy_result["final_state"])

print("\nTotal Cost:", greedy_result["final_state"]["cost"])

print("Nodes Explored:", greedy_result["nodes_explored"])

print("Execution Time:", round(greedy_time, 6), "seconds")

===== GREEDY BEST-FIRST SEARCH RESULT =====

Actions Taken:
-> Start Washing Machine
-> Start Dishwasher

Final State:
{'time': 18, 'washing_machine': True, 'dishwasher': True, 'lights': True, 'cost': 27}

Total Cost: 27
Nodes Explored: 3
Execution Time: 0.000159 seconds


In [ ]:
print("="*95)
print(f"{'Algorithm':<20}{'Path Length':<15}{'Cost':<10}{'Nodes':<10}{'Time(s)':<12}")
print("="*95)

print(f"{'BFS':<20}{len(bfs_result['path']):<15}{bfs_result['final_state']['cost']:<10}{bfs_result['nodes_explored']:<10}{bfs_time:.6f}")

print(f"{'DFS':<20}{len(dfs_result['path']):<15}{dfs_result['final_state']['cost']:<10}{dfs_result['nodes_explored']:<10}{dfs_time:.6f}")

print(f"{'Greedy BFS':<20}{len(greedy_result['path']):<15}{greedy_result['final_state']['cost']:<10}{greedy_result['nodes_explored']:<10}{greedy_time:.6f}")

print(f"{'A*':<20}{len(astar_result['path']):<15}{astar_result['final_state']['cost']:<10}{astar_result['nodes_explored']:<10}{astar_time:.6f}")

print("="*95)

Algorithm           Path Length    Cost      Nodes     Time(s)     
BFS                 2              27        6         0.000114
DFS                 7              14        8         0.000113
Greedy BFS          2              27        3         0.000159
A*                  4              14        41        0.000221


In [ ]:
test_cases = [

    {
        "name":"Peak Hour",

        "state":{
            "time":18,
            "peak_hour":True,
            "temperature":35,
            "occupancy":3,

            "ac":True,
            "refrigerator":True,

            "washing_machine":False,
            "dishwasher":False,
            "lights":True,

            "cost":0
        }
    },

    {
        "name":"Off Peak",

        "state":{
            "time":21,
            "peak_hour":False,
            "temperature":28,
            "occupancy":2,

            "ac":False,
            "refrigerator":True,

            "washing_machine":False,
            "dishwasher":False,
            "lights":True,

            "cost":0
        }
    },

    {
        "name":"Only Dishwasher Pending",

        "state":{
            "time":19,
            "peak_hour":True,
            "temperature":33,
            "occupancy":4,

            "ac":True,
            "refrigerator":True,

            "washing_machine":True,
            "dishwasher":False,
            "lights":True,

            "cost":0
        }
    }

]

In [ ]:
import pandas as pd
import time

results = []

for case in test_cases:

    state = case["state"]

    start = time.time()
    bfs_result = bfs(state)
    bfs_time = time.time()-start

    start = time.time()
    dfs_result = dfs(state)
    dfs_time = time.time()-start

    start = time.time()
    greedy_result = greedy_best_first_search(state)
    greedy_time = time.time()-start

    start = time.time()
    astar_result = astar(state)
    astar_time = time.time()-start

    results.append([
        case["name"],
        "BFS",
        bfs_result["final_state"]["cost"],
        bfs_result["nodes_explored"],
        round(bfs_time,6)
    ])

    results.append([
        case["name"],
        "DFS",
        dfs_result["final_state"]["cost"],
        dfs_result["nodes_explored"],
        round(dfs_time,6)
    ])

    results.append([
        case["name"],
        "Greedy BFS",
        greedy_result["final_state"]["cost"],
        greedy_result["nodes_explored"],
        round(greedy_time,6)
    ])

    results.append([
        case["name"],
        "A*",
        astar_result["final_state"]["cost"],
        astar_result["nodes_explored"],
        round(astar_time,6)
    ])

comparison = pd.DataFrame(
    results,
    columns=[
        "Scenario",
        "Algorithm",
        "Cost",
        "Nodes Explored",
        "Execution Time (s)"
    ]
)

comparison

,Scenario,Algorithm,Cost,Nodes Explored,Execution Time (s)
0,Peak Hour,BFS,27,6,0.000061
1,Peak Hour,DFS,14,8,0.000038
2,Peak Hour,Greedy BFS,27,3,0.000031
3,Peak Hour,A*,14,38,0.000198
4,Off Peak,BFS,14,6,0.000026
5,Off Peak,DFS,14,5,0.000014
6,Off Peak,Greedy BFS,14,3,0.000019
7,Off Peak,A*,14,22,0.000046
8,Only Dishwasher Pending,BFS,12,2,0.000005
9,Only Dishwasher Pending,DFS,6,6,0.000010


In [ ]:
def print_solution(result):

    print("Initial State")

    print()

    for step in result["path"]:

        print("↓")

        print(step)

    print()

    print("Goal State Reached")

In [ ]:
print_solution(astar_result)

Initial State

↓
Advance Time
↓
Start Dishwasher

Goal State Reached
